# Resumable T2SMark SD3.5 one-unit GPU canary
Unexecuted experiment handoff only. The project runner owns method, checkpoint, and Drive state.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Checkout and dependency setup
Resolve branch HEAD once and immediately detach it.

In [ ]:
import os, pathlib, subprocess, sys
from google.colab import userdata
assert __import__('torch').cuda.is_available(), 'CUDA is required; do not substitute a weaker model.'
checkout = pathlib.Path('/content/CEG-WM-Baseline-V1')
subprocess.run(['git','clone','--branch','Baseline-V1','https://github.com/RICHAAARC/CEG-WM.git',str(checkout)],check=True)
resolved_exact = subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip()
subprocess.run(['git','-C',str(checkout),'checkout','--detach',resolved_exact],check=True)
assert subprocess.check_output(['git','-C',str(checkout),'status','--porcelain'],text=True).strip() == ''
subprocess.run([sys.executable,'-m','pip','install','-q','diffusers==0.32.0','transformers==4.45.2','accelerate==1.1.1','huggingface_hub==0.26.2','safetensors==0.4.5','sentencepiece==0.2.0'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(checkout)],check=True)

## Stable RUN_ID runner launch
Valid artifacts are reused; missing, corrupt, or failed generation/observations are retried. Secrets enter only the child environment.

In [ ]:
RUN_ID = 't2smark_sd35_one_unit_v1'
FORCE_RERUN_ALL = False
run_dir = pathlib.Path('/content/drive/MyDrive/CEG-WM/Baseline-V1/T2SMark-Canary') / RUN_ID
child_env = dict(os.environ); child_env['HF_TOKEN'] = userdata.get('HF_TOKEN') or ''
command = [sys.executable,'-m','cegwm.baselines.t2smark_canary','--run-dir',str(run_dir),'--run-id',RUN_ID,'--project-exact',resolved_exact]
if FORCE_RERUN_ALL: command.append('--force-rerun-all')
subprocess.run(command, cwd=checkout, env=child_env, check=True)

## Checks / next step
Twelve real scores are an engineering canary only; no threshold, TPR/FPR, robustness, or paper conclusion.